## Data Ingestion and Sourcing Architecture

### Task 1.1: Environment Initialization and Configuration

#### I. IMPORTING ALL DEPENDENCIES I NEED FOR THIS PROJECT

In [ ]:
import matplotlib.pyplot as plt
import missingno as msno
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.graphics.correlation as sgc
import statsmodels.stats.api as sms
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from statsmodels.graphics.gofplots import qqplot
from statsmodels.stats.outliers_influence import OLSInfluence

### II. CONNECTION TO MY DATABASE ON POSTGRES

In [ ]:
import os
import sys

import pandas as pd

# Tells Python to look one folder up (the root folder)
sys.path.append(os.path.abspath(os.path.join("..")))


from db_connect import connect_to_db

# Connect to the database
conn = connect_to_db()

if conn:
    # Query database tables natively into Pandas
    query = "SELECT * FROM youtube_data_schema.vw_final_modeling_ready;"
    df = pd.read_sql_query(query, conn)

    print("Database connection successful! Previewing data:")
    display(df.head())

    # Always remember to close connections when done
    conn.close()

In [ ]:
df

In [ ]:
msno.matrix(df)

In [ ]:
kenya = df.loc[df["video_trending_country"] == "Kenya"]
kenya

In [ ]:
kenya = df.loc[df["channel_country"] == "Kenya"]
kenya

### Translating from foreign language to English.

In [ ]:
import gc
import os
import sys
from urllib.parse import quote_plus

import pandas as pd
import torch
from langdetect import LangDetectException, detect
from sqlalchemy import create_engine
from tqdm import tqdm
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

sys.path.append(os.path.abspath(os.path.join("..")))
from db_connect import connect_to_db

conn = connect_to_db()

if conn:
    print("🔌 Extracting translation targets from PostgreSQL...")
    query = "SELECT * FROM youtube_data_schema.vw_translation_queue;"
    df_queue = pd.read_sql_query(query, conn)
    conn.close()  # done reading, close early

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🤖 Initializing translation hardware backend on: [{device.upper()}]")

    model_name = "facebook/m2m100_418M"
    tokenizer = M2M100Tokenizer.from_pretrained(model_name)
    model = M2M100ForConditionalGeneration.from_pretrained(model_name).to(device)
    model.eval()

    M2M100_LANGS = set(tokenizer.lang_code_to_id.keys())

    print(f"🔮 Processing {len(df_queue)} videos through batch translation pipeline...")

    def clean_text(text):
        return str(text).strip() if text else ""

    titles = df_queue["video_title"].apply(clean_text).tolist()
    descriptions = df_queue["video_description"].apply(clean_text).tolist()

    title_masks = [(t and t != "[No Title]") for t in titles]
    desc_masks = [(d and d != "No description" and d != "[No Desc]") for d in descriptions]

    def detect_lang(text):
        try:
            code = detect(text)
        except LangDetectException:
            return None
        if code.startswith("zh"):
            code = "zh"
        return code if code in M2M100_LANGS else None

    batch_size = 16

    def translate_texts(texts, mask, max_length, desc_label):
        results = list(texts)
        candidates = [i for i, m in enumerate(mask) if m]
        lang_by_idx = {}
        for i in candidates:
            lang = detect_lang(texts[i])
            if lang and lang != "en":
                lang_by_idx[i] = lang

        groups = {}
        for i, lang in lang_by_idx.items():
            groups.setdefault(lang, []).append(i)

        for lang, idxs in tqdm(groups.items(), desc=f"{desc_label} (by language)"):
            tokenizer.src_lang = lang
            for start in range(0, len(idxs), batch_size):
                batch_idxs = idxs[start : start + batch_size]
                batch_texts = [texts[i] for i in batch_idxs]

                encoded = tokenizer(
                    batch_texts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                ).to(device)

                with torch.no_grad():
                    generated = model.generate(
                        **encoded,
                        forced_bos_token_id=tokenizer.get_lang_id("en"),
                        max_length=max_length,
                        num_beams=1,
                    )

                decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)

                for i, translated in zip(batch_idxs, decoded):
                    results[i] = translated

                del encoded, generated
                if device == "cuda":
                    torch.cuda.empty_cache()

        return results

    translated_titles = translate_texts(
        titles, title_masks, max_length=128, desc_label="Translating titles"
    )
    translated_titles = [
        t if m else (t if t else "[No Title]") for t, m in zip(translated_titles, title_masks)
    ]

    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    translated_descriptions = translate_texts(
        descriptions, desc_masks, max_length=256, desc_label="Translating descriptions"
    )
    translated_descriptions = [
        d if m else "No description" for d, m in zip(translated_descriptions, desc_masks)
    ]

    df_queue["video_title_en"] = translated_titles
    df_queue["video_description_en"] = translated_descriptions

    print("✅ Translation complete. df_queue is ready for writing to the DB.")
else:
    print("❌ Failed to connect to database for read.")

In [ ]:
import os
from urllib.parse import quote_plus

from sqlalchemy import create_engine

db_user = quote_plus(os.getenv("DB_USER"))
db_password = quote_plus(os.getenv("DB_PASSWORD"))
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

db_url = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
engine = create_engine(db_url)

df_queue[["video_id", "video_title_en", "video_description_en"]].to_sql(
    name="translated_dictionary",
    con=engine,
    schema="youtube_data_schema",
    if_exists="replace",
    index=False,
)
print("✨ Successfully built and populated local translation table!")
engine.dispose()

In [ ]:
import os
from urllib.parse import quote_plus

import pandas as pd
from sqlalchemy import create_engine

db_url = f"postgresql://{quote_plus(os.getenv('DB_USER'))}:{quote_plus(os.getenv('DB_PASSWORD'))}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(db_url)

query = """
SELECT
    q.video_id,
    q.video_title        AS original_title,
    t.video_title_en     AS translated_title,
    q.video_description  AS original_description,
    t.video_description_en AS translated_description
FROM youtube_data_schema.vw_translation_queue q
LEFT JOIN youtube_data_schema.translated_dictionary t
    ON q.video_id = t.video_id
LIMIT 20;
"""

df_compare = pd.read_sql_query(query, engine)
df_compare

In [ ]:
# 1. How many rows exist in each?
count_query = """
SELECT
    (SELECT COUNT(*) FROM youtube_data_schema.vw_translation_queue)  AS queue_count,
    (SELECT COUNT(*) FROM youtube_data_schema.translated_dictionary)  AS translated_count;
"""
pd.read_sql_query(count_query, engine)

In [ ]:
# 2. Any videos in the queue that are MISSING from the translated table?
missing_query = """
SELECT q.video_id, q.video_title
FROM youtube_data_schema.vw_translation_queue q
LEFT JOIN youtube_data_schema.translated_dictionary t
    ON q.video_id = t.video_id
WHERE t.video_id IS NULL;
"""
df_missing = pd.read_sql_query(missing_query, engine)
print(f"Missing translations: {len(df_missing)}")
df_missing

In [ ]:
# Get the ACTUAL count of unchanged titles (no LIMIT)
unchanged_count_query = """
SELECT COUNT(*) as unchanged_count
FROM youtube_data_schema.vw_translation_queue q
JOIN youtube_data_schema.translated_dictionary t
    ON q.video_id = t.video_id
WHERE q.video_title = t.video_title_en
  AND q.video_title NOT IN ('[No Title]', '')
"""
pd.read_sql_query(unchanged_count_query, engine)

In [ ]:
# What languages are the "unchanged" titles in?
lang_check_query = """
SELECT q.video_id, q.video_title, t.video_title_en
FROM youtube_data_schema.vw_translation_queue q
JOIN youtube_data_schema.translated_dictionary t
    ON q.video_id = t.video_id
WHERE q.video_title = t.video_title_en
  AND q.video_title NOT IN ('[No Title]', '')
"""
df_unchanged_full = pd.read_sql_query(lang_check_query, engine)

# Detect language of each unchanged title
from langdetect import LangDetectException, detect


def detect_lang_safe(text):
    try:
        return detect(str(text))
    except LangDetectException:
        return "unknown"


df_unchanged_full["detected_lang"] = df_unchanged_full["video_title"].apply(detect_lang_safe)
df_unchanged_full["detected_lang"].value_counts()

In [ ]:
# Pull only the untranslated non-English rows back out
df_retry = df_unchanged_full[df_unchanged_full["detected_lang"] != "en"].copy()
print(f"Rows to retry: {len(df_retry)}")

retry_titles = df_retry["video_title"].tolist()
retry_masks = [True] * len(retry_titles)  # all need translation

# Force-translate them using their detected language directly
# (bypassing langdetect since we already have the lang)


def translate_with_known_lang(texts, langs, max_length=128):
    results = list(texts)
    groups = {}
    for i, lang in enumerate(langs):
        if lang != "unknown" and lang in M2M100_LANGS:
            groups.setdefault(lang, []).append(i)

    for lang, idxs in tqdm(groups.items(), desc="Retrying missed translations"):
        tokenizer.src_lang = lang
        for start in range(0, len(idxs), batch_size):
            batch_idxs = idxs[start : start + batch_size]
            batch_texts = [texts[i] for i in batch_idxs]

            encoded = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length,
            ).to(device)

            with torch.no_grad():
                generated = model.generate(
                    **encoded,
                    forced_bos_token_id=tokenizer.get_lang_id("en"),
                    max_length=max_length,
                    num_beams=1,
                )

            decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
            for i, translated in zip(batch_idxs, decoded):
                results[i] = translated

            del encoded, generated
            if device == "cuda":
                torch.cuda.empty_cache()

    return results


retry_langs = df_retry["detected_lang"].tolist()
retranslated = translate_with_known_lang(retry_titles, retry_langs)
df_retry["video_title_en"] = retranslated

In [ ]:
# 1. Diagnose where the duplicates are
print("df_queue dupes:", df_queue.index.duplicated().sum())
print("df_retry dupes:", df_retry.index.duplicated().sum())

In [ ]:
# 1. Reset index first (in case set_index was partially applied)
df_queue = df_queue.reset_index()
df_retry = df_retry.reset_index()

# 2. Check what's actually duplicated in df_queue before dropping
print("Sample duplicate video_ids in df_queue:")
print(
    df_queue[df_queue.duplicated(subset="video_id", keep=False)][["video_id", "video_title"]].head(
        10
    )
)

In [ ]:
# See exactly how many times each dupe appears and spot the pattern
dupe_query = """
SELECT video_id, COUNT(*) as appearances
FROM youtube_data_schema.vw_translation_queue
GROUP BY video_id
HAVING COUNT(*) > 1
ORDER BY appearances DESC
LIMIT 20;
"""
pd.read_sql_query(dupe_query, engine)

In [ ]:
view_def_query = """
SELECT view_definition
FROM information_schema.views
WHERE table_schema = 'youtube_data_schema'
  AND table_name = 'vw_translation_queue';
"""
result = pd.read_sql_query(view_def_query, engine)
print(result["view_definition"].iloc[0])

In [ ]:
dupe_detail_query = """
SELECT 
    video_id,
    COUNT(DISTINCT video_title)       AS distinct_titles,
    COUNT(DISTINCT video_description) AS distinct_descriptions,
    COUNT(DISTINCT channel_title)     AS distinct_channels,
    COUNT(*) AS total_rows
FROM youtube_data_schema.vw_base_clean
WHERE video_title <> '[No Title]'
GROUP BY video_id
HAVING COUNT(*) > 1
ORDER BY total_rows DESC
LIMIT 20;
"""
pd.read_sql_query(dupe_detail_query, engine)

In [ ]:
# Merge retranslated titles back into df_queue
df_queue.set_index("video_id", inplace=True)
df_retry.set_index("video_id", inplace=True)
df_queue.update(df_retry[["video_title_en"]])
df_queue.reset_index(inplace=True)

# Write updated table back to DB
df_queue[["video_id", "video_title_en", "video_description_en"]].to_sql(
    name="translated_dictionary",
    con=engine,
    schema="youtube_data_schema",
    if_exists="replace",
    index=False,
)
print("✅ Retry patch written to DB.")